## Introduction

In this second laboratory we will gain some experience working with Transformer models for a variety tasks using (mostly) the Hugging Face Ecosystem. 


---
### Exercise 1: Sentiment Analysis (warm up)

In this first exercise we will start from a pre-trained BERT transformer and build up a model able to perform text sentiment analysis. Transformers are complex beasts, so we will build up our pipeline in several explorative and incremental steps.

#### Exercise 1.1: Loading the Dataset Splits
There are a many sentiment analysis datasets, but we will use one of the smallest ones available: the [Cornell Rotten Tomatoes movie review dataset](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes), which consists of 5,331 positive and 5,331 negative processed sentences from the Rotten Tomatoes movie reviews.

**Your first task**: Load the dataset and figure out what splits are available and how to get them. Spend some time exploring the dataset to see how it is organized. Note that we will be using the [HuggingFace Datasets](https://huggingface.co/docs/datasets/en/index) library for downloading, accessing, splitting, and batching data for training and evaluation.

In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
from datasets import load_dataset, get_dataset_split_names,load_dataset_builder

ds_builder = load_dataset_builder("cornell-movie-review-data/rotten_tomatoes")

We get every information possible here **without downloading it**

In [3]:
ds_builder.info

DatasetInfo(features={'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}, builder_name='parquet', dataset_name='rotten_tomatoes', config_name='default', version=0.0.0, splits={'train': SplitInfo(name='train', num_bytes=1075873, num_examples=8530, dataset_name='rotten_tomatoes'), 'validation': SplitInfo(name='validation', num_bytes=134809, num_examples=1066, dataset_name='rotten_tomatoes'), 'test': SplitInfo(name='test', num_bytes=136102, num_examples=1066, dataset_name='rotten_tomatoes')}, download_size=881052, dataset_size=1346784, size_in_bytes=2227836)

Below I extract the features field, which are essentially the types of the training points and the labels.

In [4]:
print("Features description:\n",ds_builder.info.features)

Features description:
 {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


Now we load our dataset consisting of all three splits.

In [5]:
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [6]:
ds_train = dataset["train"]
ds_val = dataset["validation"]
ds_test = dataset["test"]

Now let's inspect **ds_train object**

In [7]:
import numpy as np
indices = np.random.choice(len(ds_train), 6, replace=False)
print("The following example are structure as follows \n <label> : <phrase>\n")
for index in indices:
    print(ds_train[index]["label"],":",ds_train[index]["text"])

The following example are structure as follows 
 <label> : <phrase>

1 : as gory as the scenes of torture and self-mutilation may be , they are pitted against shimmering cinematography that lends the setting the ethereal beauty of an asian landscape painting .
0 : too often , the viewer isn't reacting to humor so much as they are wincing back in repugnance .
1 : a highly intriguing thriller , coupled with some ingenious plot devices and some lavishly built settings . . it's a worthwhile tutorial in quantum physics and slash-dash
1 : a work of extraordinary journalism , but it is also a work of deft and subtle poetry .
1 : a fantastically vital movie that manages to invest real humor , sensuality , and sympathy into a story about two adolescent boys .
0 : children and adults enamored of all things pokemon won't be disappointed .



---
### Exercise 1.2: A Pre-trained BERT and Tokenizer

The model we will use is a *very* small BERT transformer called [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) this model was trained (using self-supervised learning) on the same corpus as BERT but using the full BERT base model as a *teacher*.

**Your next task**: Load the DistilBERT model and corresponding tokenizer. Use the tokenizer on a few samples from the dataset and pass the tokens through the model to see what outputs are provided. I suggest you use the [`AutoModel`](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html) class (and the `from_pretrained()` method) to load the model and `AutoTokenizer` to load the tokenizer).

In [ ]:
from transformers import AutoTokenizer, AutoModel

model = AutoModel.from_pretrained('distilbert/distilbert-base-uncased')
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')

In [9]:
model

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

In [10]:
type(tokenizer)

transformers.models.bert.tokenization_bert.BertTokenizer

In [11]:
tokenized_sentence = tokenizer.encode(ds_train[0]["text"])
len(tokenized_sentence)

## now let's try to tokenize more than one frase
tokens = tokenizer.encode([ds_train[0]["text"],ds_train[1]["text"]])
len(tokens)

2

Let's verify the **lenght of each sentence**

In [12]:
for token in tokens :
    print(len(token))

47
52


We can also obtain the tensor, but when working with tensors we need to use padding, otherwise **we cannot process the batch**.

In [13]:
tokenized_sentence = tokenizer.encode([ds_train[0]["text"],ds_train[1]["text"]],return_tensors='pt',padding=True)
tokenized_sentence.shape

torch.Size([2, 52])

Now we can feed the model our tokenized sentences, but first let’s create a function to actually **collect the sentences**.

In [14]:
def collect_tokenized_tensor(phrases: list[str] | str, tokenizer, num_sample=None):
    if isinstance(phrases, str):
        phrases = [phrases]

    if num_sample is not None and num_sample < len(phrases):
        indices = np.random.choice(len(phrases), num_sample, replace=False)
        phrases = [phrases[i] for i in indices]

    return tokenizer.encode(phrases,return_tensors="pt",padding=True),indices

In [15]:
input_tensors , indices  = collect_tokenized_tensor(ds_train["text"],tokenizer,num_sample= 2)
print(input_tensors.shape)
indices

torch.Size([2, 36])


array([ 190, 7189])

In [16]:
out = model(input_tensors)
type(out)

transformers.modeling_outputs.BaseModelOutput

In [17]:
dir(out)

['__annotations__',
 '__class__',
 '__class_getitem__',
 '__contains__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__ior__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__post_init__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__reversed__',
 '__ror__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'attentions',
 'clear',
 'copy',
 'fromkeys',
 'get',
 'hidden_states',
 'items',
 'keys',
 'last_hidden_state',
 'move_to_end',
 'pop',
 'popitem',
 'setdefault',
 'to_tuple',
 'update',
 'values']

In [18]:
out.last_hidden_state.shape

torch.Size([2, 36, 768])

In [19]:
batch = tokenizer(ds_train[indices]['text'], return_tensors='pt', padding=True)
batch

{'input_ids': tensor([[  101,  2336,  2089,  2025,  3305,  2673,  2008,  6433,  1011,  1011,
          1045,  1005,  1049,  2025,  2469,  2130,  2771,  3148, 18637,  2370,
          2515,  1011,  1011,  2021,  2027,  2097,  2471,  5121,  2022, 15677,
          1010,  1998, 17319, 15936,  1012,   102],
        [  101,  1037,  7046,  6209,  2091,  2121,  2084,  2115,  2203,  1011,
          1997,  1011,  2095, 22649,  1006,  1047,  1007,  4861,  1012,   102,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 

The **attention mask** tells the model that it should **ignore those tokens** during the attention computation.

In [78]:
print(type(batch))
tokenizer.decode(token_ids = batch['input_ids'])

<class 'transformers.tokenization_utils_base.BatchEncoding'>


["[CLS] sure, it ' s contrived and predictable, but its performances are so well tuned that the film comes off winningly, even though it ' s never as solid as you want it to be. [SEP]",
 '[CLS] sits uneasily as a horror picture... but finds surprising depth in its look at the binds of a small family. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]']

Now I want to show that the two output tensors actually change when we take the attention mask into account, using ```torch.allclose() ```

In [79]:
no_masking = model.forward(input_ids=batch['input_ids']).last_hidden_state
print(no_masking)
assert torch.allclose(no_masking, out.last_hidden_state, atol=1e-5), "Tensor are NOT equal"

masking = model.forward(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).last_hidden_state
print(torch.allclose(masking, out.last_hidden_state, atol=1e-5))

## equivantly i can use
_masking = model.forward(**batch).last_hidden_state
print(torch.allclose(_masking, out.last_hidden_state, atol=1e-5))


tensor([[[ 0.1350, -0.1764,  0.0227,  ..., -0.0139,  0.4577,  0.3199],
         [ 1.1633, -0.1951, -0.1057,  ...,  0.1433,  1.1811,  0.1909],
         [-0.1436,  0.1702,  0.3647,  ..., -0.1722,  0.7115,  0.3971],
         ...,
         [-0.0278, -0.3496, -0.0714,  ..., -0.0255,  0.3467,  0.4527],
         [ 0.8219,  0.0868, -0.2726,  ...,  0.2394, -0.4352, -0.4377],
         [ 0.1310, -0.1578,  0.4093,  ...,  0.3165,  0.5319, -0.2550]],

        [[-0.2410, -0.3474,  0.0184,  ...,  0.0217,  0.5432,  0.1660],
         [-0.0075,  0.0802,  0.2544,  ..., -0.1497,  0.5396,  0.2487],
         [-0.3519, -0.0173,  0.3409,  ..., -0.2002,  0.5217,  0.2427],
         ...,
         [ 0.2900, -0.1403,  0.5508,  ..., -0.0552,  0.3590, -0.2440],
         [ 0.3172, -0.1474,  0.5664,  ..., -0.0338,  0.3321, -0.2909],
         [ 0.2726, -0.2354,  0.5036,  ...,  0.0068,  0.3039, -0.3358]]],
       grad_fn=<NativeLayerNormBackward0>)
False
False


We get **False** as expected


---
### Exercise 1.3: A Stable Baseline

In this exercise I want you to:
1. Use DistilBERT as a *feature extractor* to extract representations of the text strings from the dataset splits;
2. Train a classifier (your choice, by an SVM from Scikit-learn is an easy choice).
3. Evaluate performance on the validation and test splits.

These results are our *stable baseline* -- the **starting** point on which we will (hopefully) improve in the next exercise.

**Hint**: There are a number of ways to implement the feature extractor, but probably the best is to use a [feature extraction `pipeline`](https://huggingface.co/tasks/feature-extraction). You will need to interpret the output of the pipeline and extract only the `[CLS]` token from the *last* transformer layer. *How can you figure out which output that is?*

We construct a function to directly extract the CLS features. The CLS feature vector is located at `feats[i][0][0]` because `feats` has shape `(num_sentences, max_words_in_all_sentences, 1 + vect_dim)`.

In [80]:
from transformers import pipeline
import torch
from sklearn.svm import SVC
from sklearn.metrics import classification_report

def  extract_features(model,tokenizer,device,ds,batch_size=128):
    feature_extractor = pipeline("feature-extraction",framework="pt", model=model, tokenizer=tokenizer, 
                                device=device)
    feats = feature_extractor(list(ds["text"]), return_tensors='pt', batch_size=batch_size)
    cls_feats = torch.vstack([feats[i][0][0] for i in range(len(feats))])
    return cls_feats

Next, we apply an **SVM classifier** to the sentence representations, which requires converting the tensor into a NumPy array.

In [81]:
## feature extractions
cls_feats_train = extract_features(model=model,tokenizer=tokenizer,device=device,ds=ds_train).cpu().detach().numpy()
cls_feats_val = extract_features(model=model,tokenizer=tokenizer,device=device,ds=ds_val).cpu().detach().numpy()
cls_feats_test = extract_features(model=model,tokenizer=tokenizer,device=device,ds=ds_test).cpu().detach().numpy()

label_train = np.array(ds_train['label'])
label_val = np.array(ds_val['label'])
label_test = np.array(ds_test['label'])

svc = SVC(kernel='linear')
fitted_svc = svc.fit(cls_feats_train,label_train)

In [82]:
print(classification_report(label_val,svc.predict(cls_feats_val)))

              precision    recall  f1-score   support

           0       0.80      0.85      0.82       533
           1       0.84      0.79      0.81       533

    accuracy                           0.82      1066
   macro avg       0.82      0.82      0.82      1066
weighted avg       0.82      0.82      0.82      1066



In [83]:
print(classification_report(label_test,svc.predict(cls_feats_test)))

              precision    recall  f1-score   support

           0       0.80      0.82      0.81       533
           1       0.82      0.79      0.80       533

    accuracy                           0.81      1066
   macro avg       0.81      0.81      0.81      1066
weighted avg       0.81      0.81      0.81      1066



As a baseline, we take the F1‑scores of **0.82 on validation** and **0.81 on test**. The small gap indicates that the model is not overfitting, and the approach perform **well out of the box**.

---
---
## Exercise 2: Fine-tuning DistilBERT

In this exercise we will fine-tune the DistilBERT model to (hopefully) improve sentiment analysis performance.


---
### Exercise 2.1: Token Preprocessing

The first thing we need to do is *tokenize* our dataset splits -- we don't want to re-tokenize our inputs for every batch! Our current datasets return a dictionary with *strings*, but we want *input token ids* (i.e. the output of the tokenizer). This is easy enough to do by hand, but the Hugging Face `Dataset` class provides convenient, efficient, and *lazy* methods. See the documentation for [`Dataset.map`](https://huggingface.co/docs/datasets/v3.5.0/en/package_reference/main_classes#datasets.Dataset.map).

**Tip**: Verify that your new datasets are returning for every element: `text`, `label`, `intput_ids`, and `attention_mask`.

In [84]:
# 1. Definisci la funzione che applicheremo a ogni "blocco" di dati
def tokenize_function(ds):
        return tokenizer(ds["text"], truncation=True, padding=True,return_tenors='pt')

tokenized_ds_train = ds_train.map(tokenize_function, batched=True)
print(tokenized_ds_train.column_names)

tokenized_ds_val = ds_val.map(tokenize_function,batched = True)
tokenized_ds_test = ds_test.map(tokenize_function,batched = True)

['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']



---
### Exercise 2.2: Setting up the Model to be Fine-tuned

In this exercise we need to prepare the base Distilbert model for fine-tuning for a *sequence classification task*. This means, at the very least, appending a new, randomly-initialized classification head connected to the `[CLS]` token of the last transformer layer. Luckily, HuggingFace already provides an `AutoModel` for just this type of instantiation: [`AutoModelForSequenceClassification`](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html#automodelforsequenceclassification). You will want you instantiate one of these for fine-tuning.

In [ ]:
from transformers import DistilBertForSequenceClassification

cls_model = DistilBertForSequenceClassification.from_pretrained('distilbert/distilbert-base-uncased', num_labels=2) 

In [86]:
cls_model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

Now let's inspect the **output of the model**

In [87]:
tokens = tokenizer(ds_train[2:4]['text'], return_tensors='pt', padding=True, truncation=True)
out = cls_model(**tokens)
out

SequenceClassifierOutput(loss=None, logits=tensor([[ 0.1938, -0.0509],
        [ 0.1925, -0.0305]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

From the logits, we can see that both sentences are assigned to the negative class (before training) 


---
### Exercise 2.3: Fine-tuning DistilBERT

Finally. In this exercise you should use a HuggingFace [`Trainer`](https://huggingface.co/docs/transformers/main/en/trainer) to fine-tune your model on the Rotten Tomatoes training split. Setting up the trainer will involve (at least):


1. Instantiating a [`DataCollatorWithPadding`](https://huggingface.co/docs/transformers/en/main_classes/data_collator) object which is what *actually* does your batch construction (by padding all sequences to the same length).
2. Writing an *evaluation function* that will measure the classification accuracy. This function takes a single argument which is a tuple containing `(logits, labels)` which you should use to compute classification accuracy (and maybe other metrics like F1 score, precision, recall) and return a `dict` with these metrics.  
3. Instantiating a [`TrainingArguments`](https://huggingface.co/docs/transformers/v4.51.1/en/main_classes/trainer#transformers.TrainingArguments) object using some reasonable defaults.
4. Instantiating a `Trainer` object using your train and validation splits, you data collator, and function to compute performance metrics.
5. Calling `trainer.train()`, waiting, waiting some more, and then calling `trainer.evaluate()` to see how it did.

**Tip**: When prototyping this laboratory I discovered the HuggingFace [Evaluate library](https://huggingface.co/docs/evaluate/en/index) which provides evaluation metrics. However I found it to have insufferable layers of abstraction and getting actual metrics computed. I suggest just using the Scikit-learn metrics...

In [88]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
data_collator

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distill-bert-fine-tune",
    gradient_accumulation_steps=8,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,                 
    gradient_checkpointing=True,
    learning_rate=1e-3,               
    num_train_epochs=4,              
    per_device_train_batch_size=64,   
    per_device_eval_batch_size=64,    
    weight_decay=0.01,                
    eval_strategy="epoch",            
    save_strategy="epoch",            
    logging_steps=15,                
    load_best_model_at_end=True,
    bf16=True        
)

- **`learning_rate=1e-3`**  
  Sets the initial learning rate. This is relatively high for Transformers, but combined with warm‑up and cosine decay it can still work.

- **`load_best_model_at_end=True`**  
  Automatically reloads the best checkpoint (based on evaluation metric) after training.

Now I define the ```compute_metrics``` function, which will later be passed to the Trainer object. Inside this function, I compute **accuracy**, **F1‑score**, **precision**, and **recall** (using sklearn.metrics functions).

In [90]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted')
    
    return {
        "accuracy": acc, 
        "f1": f1, 
        "precision": precision, 
        "recall": recall
    }

In [102]:
from transformers import Trainer

trainer = Trainer(
    model=cls_model,                       
    args=training_args,                
    train_dataset=tokenized_ds_train,  
    eval_dataset=tokenized_ds_val,   
    processing_class=tokenizer,         
    compute_metrics=compute_metrics     
)
trainer

Now let's **proceed with the training**...

In [103]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,4.570601,0.495120,0.783302,0.782493,0.787579,0.783302
2,2.301255,0.448100,0.821764,0.821754,0.821836,0.821764
3,1.140416,0.469018,0.838649,0.838567,0.839337,0.838649
4,0.569031,0.717862,0.837711,0.837659,0.838141,0.837711


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=68, training_loss=1.9342362705399008, metrics={'train_runtime': 299.7882, 'train_samples_per_second': 113.814, 'train_steps_per_second': 0.227, 'total_flos': 688561398607680.0, 'train_loss': 1.9342362705399008, 'epoch': 4.0})

All metrics are **computed on the validation set**

Now we **evaluate** the model...

In [109]:
eval_result = trainer.evaluate()
eval_result

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.569031,0.448100,4,0.821764,0.821754,0.821836,0.821764


{'eval_loss': 0.4480999708175659,
 'eval_accuracy': 0.8217636022514071,
 'eval_f1': 0.8217535633446332,
 'eval_precision': 0.8218361059765866,
 'eval_recall': 0.8217636022514071}

Note: due to how the ```Trainer``` is implemented, it also outputs the last saved training loss after calling .evaluate() 

In [ ]:
test_result = trainer.predict(tokenized_ds_test)
test_result.metrics

{'test_loss': 0.44042372703552246,
 'test_accuracy': 0.8264540337711069,
 'test_f1': 0.8264196646337437,
 'test_precision': 0.8267127920412592,
 'test_recall': 0.8264540337711069,
 'test_runtime': 0.0809,
 'test_samples_per_second': 13176.264,
 'test_steps_per_second': 210.128}

I want to squeeze out a bit more performance, so I’m trying to implement a very simple **grid search based on the F1‑score**.

We use `itertools.product()`, which takes two lists and computes their Cartesian product, i.e., it produces tuples representing **every possible combination**.”

In [ ]:
import itertools
import torch
import gc
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

def grid_search(learning_rates,epochs_list,t_ds_train,t_ds_eval,tokenizer,compute_metrics,device):
    
    best_f1 = 0.0
    best_params = {}
    all_results = []

    for lr, epochs in itertools.product(learning_rates, epochs_list):
        
        print(f"Start training with = {lr} | Epochs = {epochs}")
        
        model_trial = AutoModelForSequenceClassification.from_pretrained(
            'distilbert/distilbert-base-uncased',
            num_labels=2).to(device)
        
        output_folder = f"./distill-bert-LR_{lr}-EPOCHS_{epochs}"
        
        trial_args = TrainingArguments(
            output_dir=output_folder,
            learning_rate=lr,                  
            num_train_epochs=epochs,           
            gradient_accumulation_steps=8,
            lr_scheduler_type='cosine',
            warmup_ratio=0.1,
            gradient_checkpointing=True,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_steps=15,
            load_best_model_at_end=True,
            bf16=True
            )
        
        trial_trainer = Trainer(
            model=model_trial,
            args=trial_args,
            train_dataset=t_ds_train,
            eval_dataset=t_ds_eval,
            processing_class=tokenizer,
            compute_metrics=compute_metrics
        )
        
        trial_trainer.train()

        val_results = trial_trainer.evaluate()
        f1 = val_results['eval_f1']       
        print(f"Model has been evaluated obtained with a F1-score : {f1:.4f}")
        
        all_results.append({'lr': lr, 'epochs': epochs, 'f1': f1 })
        
        if  f1 > best_f1:
            best_f1 = f1
            best_params = {'lr': lr, 'epochs': epochs}
            print("New best")
            
        del model_trial
        del trial_trainer
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\n Search Complete")

    return best_params,best_f1

In [116]:
learning_rates = [2e-5, 2e-4,2e-3]  
epochs_list = [4,5] 

best_parameters, best_f1_score = grid_search(
    learning_rates=learning_rates,
    epochs_list=epochs_list,
    t_ds_train=tokenized_ds_train,      
    t_ds_eval=tokenized_ds_val,         
    tokenizer=tokenizer,                
    compute_metrics=compute_metrics,
    device=device
)

Start training with = 2e-05 | Epochs = 4


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,3.375151,0.386024,0.829268,0.829263,0.829310,0.829268
2,2.739923,0.358110,0.845216,0.845212,0.845246,0.845216
3,2.104095,0.360200,0.848030,0.847987,0.848427,0.848030
4,2.001016,0.364223,0.847092,0.847089,0.847122,0.847092


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
5.545964,0.675558,5,0.592871,0.543046,0.664706,0.592871


Model has been evaluated obtained with a F1-score : 0.5430

 Search Complete


In [119]:
print("Best paramaters(for this little grid-search):",best_parameters,"with associated f1-score:",best_f1_score)

Best paramaters(for this little grid-search): {'lr': 2e-05, 'epochs': 4} with associated f1-score: 0.8452123544988212


Using the best configuration (`lr = 2e‑5`, `epochs = 4`), the model reaches an F1‑score of **0.845**, improving over the **baseline score of 0.81**.

---

### References
Used the sources suggested at the beginning.

### AI usage
Barely used AI in these two exercises. I only asked for suggestions on some `Trainer` settings; it recommended enabling gradient checkpointing and adjusting the gradient accumulation parameter.



---
---
## Exercise 3: Choose your Own Adventure

As promised, you should choose **one** of the following exercises to work. Well, at *least* one. If you want to do them all, that is also OK! Or if you want to propose something else as a third exercise, reach out to me on the Discord!


---
### Exercise 3.3: A Text-to-image Retrieval System (hard, but not *too* hard)

Implement a simple text-to-image retrieval system with a simple user interface --- using, for example, [gradio](https://www.gradio.app/), or [Marimo](https://marimo.io/), or [Shiny](https://shiny.posit.co/). Your application should *index* (e.g. compute visual descriptors for) a small dataset of images like [Flickr8k](https://huggingface.co/datasets/jxie/flickr8k). It should provide a user interface with which a user can enter a short text prompt (e.g. "a photo of dogs playing in the snow") and then display the top-10 matching images from the indexed dataset.

Note that there is no following code block with "Your code here" for this exercise. You will definitely want to implement this outside of a Jupyter Notebook.

**Hint**: The **CLIP** model is practically *made* for just such an application.

**Why choose this exercise?** Well, this is a course on Deep Learning *Applications*, and this is your chance to *build* one!

---

## Redirecting to Text to Image Retrieval App

As suggested, I decided to implement it outside the Jupyter notebook and in a separate repository, because I preferred to keep it independent rather than nested within this one.
All information about it can be found in the README.md file in the following repo :  [Text-to-Image-Retrieval-app](https://github.com/Andreathegm/Text-to-Image-Retrieval-app)  
or directly try the app at https://huggingface.co/spaces/Andy-6/Text-to-ImageApp